# A general type-2 variant: α-planes of a fuzzy fuzzifier

**Computes:** A general type-2 fuzzy c-means (GT2) with a triangular fuzzifier on [m_1, m_2] processed by the α-planes α = 1/3, 2/3 and 1, as described in Section S17 of the supplementary material, at the settings of IT2 on the imbalance sweep at ratio 20, Ecoli, New-thyroid, the equal-size controls of the sweep and the sixteen main sets (DIBCO excluded), compared with FCM, T1d and IT2 by ARI under label-tuned selection and the claim rule; the two-class decision against the nearest-prototype partition; a rerun at the selected settings with 1,000 iterations.

**Inputs:** the per-setting ARI of FCM, T1d and IT2 from the checkpoints of 20 (`db20_*.pkl`) or of 22 (`rob22_*.pkl`); UCI data; for New-thyroid the data file written by 10.
**Outputs:** `results/23/results_23.json`, checkpoints `gt2_23_*.pkl` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, shutil, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "23")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")
BASE = os.path.dirname(OUTDIR)
DIR_06 = os.environ.get("DIR_06", os.path.join(BASE, "06"))
DIR_07 = os.environ.get("DIR_07", os.path.join(BASE, "07"))
DIR_10 = os.environ.get("DIR_10", os.path.join(BASE, "10"))
DIR_12 = os.environ.get("DIR_12", os.path.join(BASE, "12"))
DIR_20 = os.environ.get("DIR_20", os.path.join(BASE, "20"))
DIR_22 = os.environ.get("DIR_22", os.path.join(BASE, "22"))
print(f"{len(glob.glob(os.path.join(DIR_20, 'db20_*.pkl'))):3d} checkpoints of 20, {len(glob.glob(os.path.join(DIR_22, 'rob22_*.pkl'))):3d} of 22, "
      f"{len(glob.glob(os.path.join(OUTDIR, 'gt2_23_*.pkl'))):3d} of 23")

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir="data"):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir="data"):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C

In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
def hr_bounds_minmax(D, m1, m2):
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    return np.minimum(U1, U2), np.maximum(U1, U2)

def run_it2fcm_mm(X, V0, m1, m2, mc=None, iters=300, tol=1e-6, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

MM = "IT2FCM-mm"

def run_instance07(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in IT2_KEYS}
    t0 = time.perf_counter(); mism_pts = mism_runs = 0
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for k in IT2_KEYS:
            a, b, mode = k; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
            V, hp, hn = run_it2fcm_mm(X, V0, a, b, mc=mc, tol=TOL, order=order)
            sc = evaluate(V, hp) if evaluate else scores(y, hp, C)
            d = rec[k]
            for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
            d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, hp[sub])); d["ch"].append(calinski(X, hp))
            mm_ = int((hp != hn).sum()); mism_pts += mm_; mism_runs += int(mm_ > 0)
    return dict(rec={MM: rec}, sec_per_run={MM: (time.perf_counter() - t0) / (N_INIT * len(IT2_KEYS))},
                mism_points=mism_pts, mism_runs=mism_runs)

def dibco_instance07(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M), f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance07(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
from scipy import stats
MAIN = SYN + REAL + ("dibco",)
SWEEPS = tuple(NOISE) + tuple(IMB)
FAM5 = dict(FAMILIES); FAM5[MM] = IT2_KEYS
PAIRS5 = (("IT2FCM", "FCM"), ("IT2FCM", "T1dec"),
          (MM, "FCM"), (MM, "T1dec"), (MM, "IT2FCM"), (MM, "IT2FCM-NT"))

def halves(nm, R):
    ids = sorted(R); h = len(ids) // 2
    if nm != "dibco": return ids[:h], ids[h:]
    rng = np.random.default_rng(7); years = {i: R[i]["name"][5:9] for i in ids}; fold = {}
    for yv in sorted(set(years.values())):
        ix = [i for i in ids if years[i] == yv]; ix = [ix[j] for j in rng.permutation(len(ix))]
        for j, i in enumerate(ix): fold[i] = j % 2
    return [i for i in ids if fold[i] == 0], [i for i in ids if fold[i] == 1]

def tp(d):
    d = np.asarray(d, float)
    if len(d) < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if len(d) else 0.0, t_p=1.0, w_p=1.0, n=len(d))
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, n=len(d))

def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."

def unsup_value(rec_f, metric, by):
    keys = list(rec_f); nq = len(rec_f[keys[0]][by]); vals = []
    for q in range(nq):
        k = max(keys, key=lambda kk: rec_f[kk][by][q]); vals.append(rec_f[k][metric][q])
    return float(np.mean(vals))

def merge(R06, R07):
    ids = sorted(set(R06) & set(R07)); R = {}
    for i in ids:
        R[i] = dict(R06[i]); R[i]["rec"] = dict(R06[i]["rec"]); R[i]["rec"][MM] = R07[i]["rec"][MM]
        R[i]["sec_per_run"] = dict(R06[i]["sec_per_run"]); R[i]["sec_per_run"][MM] = R07[i]["sec_per_run"][MM]
        R[i]["mism_points"] = R07[i].get("mism_points", 0); R[i]["mism_runs"] = R07[i].get("mism_runs", 0)
    return R

def analyse_dataset(nm, R, metric):
    A, B = halves(nm, R); out = {"n": len(R)}
    for by in ("sil", "ch"):
        val = {f: {i: unsup_value(R[i]["rec"][f], metric, by) for i in R} for f in FAM5}
        cmp = {}
        for (x, z) in PAIRS5:
            a = tp([val[x][i] - val[z][i] for i in A]); b = tp([val[x][i] - val[z][i] for i in B])
            cmp[f"{x} vs {z}"] = dict(A=a, B=b, verdict=verdict(a, b), mean=float(np.mean([val[x][i] - val[z][i] for i in R])))
        out[by] = dict(mean={f: float(np.mean(list(val[f].values()))) for f in FAM5}, cmp=cmp)
    lt = {}
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(FAM5[f], key=lambda k: np.mean([np.mean(R[i]["rec"][f][k][metric]) for i in tr])) for f in FAM5}
        test = {f: {i: float(np.mean(R[i]["rec"][f][sel[f]][metric])) for i in te} for f in FAM5}
        lt[sn] = dict(selected={f: str(sel[f]) for f in sel}, cmp={f"{x} vs {z}": tp([test[x][i] - test[z][i] for i in te]) for (x, z) in PAIRS5})
    out["label_tuned"] = dict(splits=lt, verdict={c: verdict(lt["A->B"]["cmp"][c], lt["B->A"]["cmp"][c]) for c in lt["A->B"]["cmp"]},
                              mean={c: float(np.mean([lt[s]["cmp"][c]["mean"] for s in lt])) for c in lt["A->B"]["cmp"]})
    out["sec_per_run"] = {f: float(np.median([R[i]["sec_per_run"][f] for i in R])) for f in FAM5}
    out["mm_partition_vs_nearest"] = dict(points=int(sum(R[i]["mism_points"] for i in R)), runs=int(sum(R[i]["mism_runs"] for i in R)))
    out["mm_fingerprint"] = float(np.mean([np.mean(R[i]["rec"][MM][k]["acc"]) for i in R for k in IT2_KEYS]))
    return out

def dataset_level(RES, names, key):
    rows = [n for n in names if n in RES]; res = {}
    for (x, z) in PAIRS5:
        c = f"{x} vs {z}"
        d = np.array([RES[n]["label_tuned"]["mean"][c] if key == "label_tuned" else RES[n][key]["cmp"][c]["mean"] for n in rows])
        w = float(stats.wilcoxon(d).pvalue) if np.any(np.abs(d) > 1e-12) else 1.0
        res[c] = dict(n=len(rows), median=float(np.median(d)), pos=int((d > 0).sum()), neg=int((d < 0).sum()), wilcoxon_p=w)
    if key != "label_tuned":
        M = np.array([[RES[n][key]["mean"][f] for f in FAM5] for n in rows])
        fr = stats.friedmanchisquare(*M.T); ranks = np.mean([stats.rankdata(-r) for r in M], axis=0)
        res["friedman"] = dict(p=float(fr.pvalue), mean_rank=dict(zip(FAM5, map(float, ranks))))
    return res

In [ ]:
MAXIT = 300
def fcm10(X, V0, m):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, fcm_membership(dist(X, V), m).argmax(1), None, it, conv

def t1d10(X, V0, ma, mb):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, dist(X, V).argmin(1), None, it, conv

def it2km10(X, V0, m1, m2, mc, order, bounds):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1), it, conv

def nt10(X, V0, m1, m2, mc):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1), None, it, conv

def fit10(fam, key, X, V0, order):
    if fam == "FCM": return fcm10(X, V0, key)
    if fam == "T1dec": return t1d10(X, V0, key[0], key[1])
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM": return it2km10(X, V0, a, b, mc, order, hr_bounds)
    if fam == MM: return it2km10(X, V0, a, b, mc, order, hr_bounds_minmax)
    return nt10(X, V0, a, b, mc)

def xie_beni(X, V, m=2.0):
    D = dist(X, V); U = fcm_membership(D, m)
    sep = min(float(((V[j] - V[k]) ** 2).sum()) for j in range(len(V)) for k in range(j + 1, len(V)))
    return float(((U ** m) * D ** 2).sum() / (len(X) * max(sep, EPS)))

def keys_for(fam, G):
    if fam == "FCM": return list(G)
    if fam == "T1dec": return [(a, b) for a in G for b in G]
    return [(a, b, mode) for i, a in enumerate(G) for b in G[i + 1:] for mode in ("m1", "mid", "m2")]

def run_instance10(X, y, C, s, fams, centres=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    fields = ["ari", "acc", "nmi", "sil", "ch", "negxb", "it", "conv"] + (["negcerr"] if centres is not None else [])
    rec = {f: {k: {fl: [] for fl in fields + (["acc_n", "ari_n"] if f == MM else [])} for k in keys} for f, keys in fams.items()}
    V_store = {f: {k: [] for k in keys} for f, keys in fams.items()}; secs = {f: 0.0 for f in fams}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in fams.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab, near, it, conv = fit10(fam, k, X, V0, order)
                sc = scores(y, lab, C); d = rec[fam][k]
                for fl in ("ari", "acc", "nmi"): d[fl].append(sc[fl])
                d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab)); d["negxb"].append(-xie_beni(X, V))
                d["it"].append(it); d["conv"].append(bool(conv))
                if centres is not None: d["negcerr"].append(-centre_error_h(V, centres))
                if fam == MM:
                    sn = scores(y, near, C); d["acc_n"].append(sn["acc"]); d["ari_n"].append(sn["ari"])
                V_store[fam][k].append(V.astype(np.float32))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, V=V_store, C=C, sec_per_run={f: secs[f] / (N_INIT * len(fams[f])) for f in fams})

In [ ]:
DATA_DIR = os.path.join(OUTDIR, "data"); os.makedirs(DATA_DIR, exist_ok=True)
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
UCI = "https://archive.ics.uci.edu/ml/machine-learning-databases/"
EXPECT = {"new_thyroid": (215, 5, 3), "dermatology": (358, 34, 6), "yeast": (1484, 8, 10), "page_blocks": (5473, 10, 5)}

def _get(url, fn):
    p = os.path.join(DATA_DIR, fn)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        urllib.request.urlretrieve(url, p)
    return p

def _parse(name, how, p):
    import pandas as pd
    if name == "new_thyroid" and how == "mirror":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, :5], d.iloc[:, 5]
    elif name == "new_thyroid":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, 1:], d.iloc[:, 0]
    elif name == "dermatology":
        d = pd.read_csv(p, header=None, na_values="?").dropna(); X, y = d.iloc[:, :34], d.iloc[:, 34]
    elif name == "yeast":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, 1:9], d.iloc[:, 9]
    elif name == "page_blocks":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, :10], d.iloc[:, 10]
    return X.to_numpy(float), y.astype(str).str.strip().to_numpy()

def _openml(name):
    import pandas as pd
    from sklearn.datasets import fetch_openml
    b = fetch_openml(name={"page_blocks": "page-blocks", "new_thyroid": "thyroid-new"}.get(name, name), version=1, as_frame=True, parser="auto")
    X = b.data.apply(pd.to_numeric, errors="coerce"); keep = ~X.isna().any(axis=1)
    return X[keep].to_numpy(float), b.target[keep].astype(str).str.strip().to_numpy()

def load_partb(name):
    cache = os.path.join(DATA_DIR, f"partb_{name}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False); return z["X"], z["y"], int(z["C"])
    tries = {"new_thyroid": [("mirror", MIRROR + "new-thyroid.csv", "new-thyroid.csv"), ("uci", UCI + "thyroid-disease/new-thyroid.data", "new-thyroid.data"), ("openml", None, None)],
             "dermatology": [("uci", UCI + "dermatology/dermatology.data", "dermatology.data"), ("openml", None, None)],
             "yeast": [("uci", UCI + "yeast/yeast.data", "yeast.data"), ("openml", None, None)],
             "page_blocks": [("uciZ", UCI + "page-blocks/page-blocks.data.Z", "page-blocks.data.Z"), ("openml", None, None)]}[name]
    errors = []
    for how, url, fn in tries:
        try:
            if how == "openml":
                Xr, yr = _openml(name)
            elif how == "uciZ":
                pz = _get(url, fn); p = pz[:-2]
                subprocess.run(f"gzip -dc '{pz}' > '{p}'", shell=True, check=True); Xr, yr = _parse(name, how, p)
            else:
                Xr, yr = _parse(name, how, _get(url, fn))
            break
        except Exception as e:
            errors.append(f"{how}: {e!r}"); Xr = None
    if Xr is None:
        raise RuntimeError(f"{name}: every source failed:\n  " + "\n  ".join(errors))
    import pandas as pd
    y = pd.factorize(yr, sort=True)[0].astype(int); C = len(np.unique(y))
    N0, M0, C0 = EXPECT[name]
    assert C == C0 and Xr.shape[1] == M0 and abs(len(Xr) - N0) <= 10, f"{name}: unexpected shape {Xr.shape}, C={C}"
    X = Xr[:, Xr.std(0) > 0]; X = (X - X.mean(0)) / X.std(0)
    np.savez(cache, X=X, y=y, C=C); return X, y, C

In [ ]:
for _f in ("sonar.csv", "pima-indians-diabetes.csv", "breast-cancer-wisconsin.data", "wheat-seeds.csv", "glass.csv",
           "ionosphere.csv", "banknote_authentication.csv", "ecoli.csv"):
    _get(MIRROR + _f, _f)
def fcma_membership(D, alpha, ma):
    z = np.log(np.maximum(alpha, 1e-300))[None, :] - (2.0 / (ma - 1.0)) * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)
def fcma10(X, V0, ma, mb, freeze_alpha=False, track=False):
    V = V0.copy(); C = len(V0); alpha = np.full(C, 1.0 / C); conv = False; J = []
    for it in range(1, MAXIT + 1):
        U = fcma_membership(dist(X, V), alpha, ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if not freeze_alpha:
            S = (W * dist(X, Vn) ** 2).sum(0); a = np.maximum(S, 1e-300) ** (1.0 / mb)
            alpha = np.maximum(a / a.sum(), 1e-12); alpha /= alpha.sum()
        if track:
            U2 = fcma_membership(dist(X, Vn), alpha, ma)
            J.append(float(((alpha ** (1 - ma))[None, :] * U2 ** ma * dist(X, Vn) ** 2).sum()))
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lab = fcma_membership(dist(X, V), alpha, ma).argmax(1)
    return (V, lab, alpha, it, conv, J) if track else (V, lab, alpha, it, conv)
def keys12(fam, G):
    return list(G) if fam == "FCMA" else [(a, b) for a in G for b in G]
def fit12(fam, key, X, V0):
    return fcma10(X, V0, key, key) if fam == "FCMA" else fcma10(X, V0, key[0], key[1])
chk = []
for nm in ("volume", "imb_C8_r20", "ecoli"):
    X, y, C = get_instance(nm, 0, DATA_DIR); V0 = init_pp(X, C, np.random.default_rng(10_000))
    for m in (1.5, 2.0, 5.0):
        Vf, labf, _, _, _ = fcm10(X, V0, m)
        Va, laba, _, _, _ = fcma10(X, V0, m, m, freeze_alpha=True)
        _, _, _, _, _, J = fcma10(X, V0, m, m, track=True)
        rises = sum(1 for a_, b_ in zip(J, J[1:]) if b_ > a_ * (1 + 1e-9) + 1e-12)
        chk.append(dict(set=nm, m=m, max_proto_diff_vs_fcm=float(np.abs(Va - Vf).max()), same_labels=bool((laba == labf).all()), objective_rises=rises, iters=len(J)))
for c in chk: print(c)
assert all(c["max_proto_diff_vs_fcm"] < 1e-8 and c["same_labels"] for c in chk), "FCMA with frozen size variables does not reproduce FCM"
LOG_CHECKS = chk
print("checks passed: frozen FCMA = FCM; objective increases:", sum(c["objective_rises"] for c in chk), "(should be 0)")

In [ ]:
G_B = (1.05, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PARTB_SETS = ("new_thyroid", "dermatology", "yeast", "page_blocks")
def grid_of(nm): return (G_B if nm in PARTB_SETS else GRID)
def n_inst(nm):
    return 14 if nm in PARTB_SETS else (N_SWEEP if (nm in IMB or nm in NOISE) else (N_SYN if nm in SYN else N_REAL))
def inst_any(nm, s):
    if nm in PARTB_SETS:
        z = np.load(os.path.join(DIR_10, "data", f"partb_{nm}.npz")); X, y, C = z["X"], z["y"], int(z["C"])
        r = np.random.default_rng(3000 + s); f = {"page_blocks": 0.3}.get(nm, 0.75)
        idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], max(2, int(round(f * (y == k).sum()))), replace=False) for k in range(C)]))
        return X[idx], y[idx], C
    return get_instance(nm, s, DATA_DIR)
def fit_any(fam, key, X, V0, order):
    if fam in ("FCMA", "FCMA-d"):
        V, lab, _, _, _ = fit12(fam, key, X, V0); return V, lab
    V, lab, _, _, _ = fit10(fam, key, X, V0, order); return V, lab
def keys_any(fam, G):
    return keys12(fam, G) if fam in ("FCMA", "FCMA-d") else keys_for(fam, G)
def sil_setup(X, s):
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    return sub, np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
def describe(X, y, lab, Cp):
    sz = np.bincount(lab, minlength=Cp); big = np.argmax(np.bincount(y))
    return int((sz <= max(2, 0.01 * len(X))).sum()), float(np.bincount(lab[y == big], minlength=Cp).max() / (y == big).sum())
def J2(X, V):
    D = dist(X, V); U = fcm_membership(D, 2.0); return float((U ** 2 * D ** 2).sum())
def ckpt(name):
    p = os.path.join(OUTDIR, name); return (pickle.load(open(p, "rb")) if os.path.exists(p) else {}), p
def save_ckpt(obj, p):
    with open(p + ".tmp", "wb") as fh: pickle.dump(obj, fh)
    os.replace(p + ".tmp", p)
def load_ref(nm):
    if nm in PARTB_SETS:
        R = pickle.load(open(os.path.join(DIR_10, f"runs10_{nm}.pkl"), "rb"))
    else:
        R = merge(pickle.load(open(os.path.join(DIR_06, f"runs06_{nm}.pkl"), "rb")), pickle.load(open(os.path.join(DIR_07, f"runs07_{nm}.pkl"), "rb")))
    p12 = os.path.join(DIR_12, f"runs12_{nm}.pkl"); R12 = pickle.load(open(p12, "rb")) if os.path.exists(p12) else {}
    return R, R12

In [ ]:
# General type-2 variant by alpha-planes, and its runs at the settings of IT2.
ALPHAS = (1.0 / 3.0, 2.0 / 3.0, 1.0)
def gt2_planes(m1, m2, alphas=ALPHAS):
    mp = 0.5 * (m1 + m2)
    return [(a, m1 + a * (mp - m1), m2 - a * (m2 - mp)) for a in alphas]
def gt2_step(X, V, order, m1, m2, mc, alphas=ALPHAS):
    D = dist(X, V); acc_V = np.zeros_like(V); acc_u = np.zeros((len(X), len(V))); wsum = 0.0
    for a, ma, mb in gt2_planes(m1, m2, alphas):
        if mb - ma < 1e-12:
            U = fcm_membership(D, ma); W = U ** mc
            Va = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS); ua = U
        else:
            lo, up = hr_bounds_minmax(D, ma, mb)
            Va, uL, uR = it2fcm_update_fast(X, order, lo, up, mc); ua = 0.5 * (uL + uR)
        acc_V += a * Va; acc_u += a * ua; wsum += a
    return acc_V / wsum, acc_u / wsum, D
def run_gt2(X, V0, m1, m2, order, mc, tol=TOL, maxit=MAXIT, alphas=ALPHAS):
    V = V0.copy(); conv = False
    for it in range(1, maxit + 1):
        Vn, _, _ = gt2_step(X, V, order, m1, m2, mc, alphas)
        if np.abs(Vn - V).max() < tol:
            V = Vn; conv = True; break
        V = Vn
    _, u, D = gt2_step(X, V, order, m1, m2, mc, alphas)
    return V, u.argmax(1), D.argmin(1), it, conv

REGIME = ["imb_C5_r20", "imb_C8_r20", "imb_C3_r20", "ecoli", "new_thyroid"]
CONTROLS = ["imb_C3_r1", "imb_C5_r1", "imb_C8_r1"]
MAIN = list(SYN) + list(REAL)
SETS = REGIME + CONTROLS + [nm for nm in MAIN if nm != "ecoli"]
if os.environ.get("SETS23"): SETS = os.environ["SETS23"].split(",")
FIELDS = ("ari", "kept", "mism", "it", "conv")
t_all = time.time()
for nm in SETS:
    if nm in PARTB_SETS and not os.path.exists(os.path.join(DIR_10, "data", f"partb_{nm}.npz")):
        DIR_10 = OUTDIR; load_partb(nm)
    out, path = ckpt(f"gt2_23_{nm}.pkl"); G = grid_of(nm); t0 = time.time()
    KEYS = [(a, b, mode) for i, a in enumerate(G) for b in G[i + 1:] for mode in ("m1", "mid", "m2")]
    for s in range(n_inst(nm)):
        if s in out: continue
        X, y, C = inst_any(nm, s); order = np.argsort(X, axis=0, kind="stable")
        rec = {k: {f: [] for f in FIELDS} for k in KEYS}
        for q in range(N_INIT):
            V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
            for k in KEYS:
                a, b, mode = k; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
                V, lab, near, it, conv = run_gt2(X, V0, a, b, order, mc)
                d = rec[k]; d["ari"].append(scores(y, lab, C)["ari"]); d["kept"].append(describe(X, y, lab, C)[1])
                d["mism"].append(int((lab != near).sum())); d["it"].append(it); d["conv"].append(bool(conv))
        out[s] = dict(C=C, N=len(X), rec=rec); save_ckpt(out, path)
    print(f"{nm:12s} {len(out)} instances in {time.time()-t0:.0f}s", flush=True)
print(f"all GT2 runs done in {time.time()-t_all:.0f}s")

In [ ]:
# Analysis: GT2 against FCM, T1d and IT2 (ARI, label-tuned on halves, claim rule), and the decision check.
FAMS = ("FCM", "T1dec", MM)
def ref_runs(nm):
    for p in (os.path.join(DIR_20, f"db20_{nm}.pkl"), os.path.join(DIR_22, f"rob22_{nm}.pkl")):
        if os.path.exists(p): return pickle.load(open(p, "rb")), os.path.basename(p)
    raise FileNotFoundError(f"{nm}: no per-setting runs of FCM, T1d and IT2 (run 20 or 22 first)")
def inst_means(get, ids, keys):
    return {k: np.array([np.mean(get(s, k)) for s in ids]) for k in keys}
def lt_cmp(inst, x, z):
    n = len(next(iter(inst[x].values()))); h = n // 2; A, B = np.arange(h), np.arange(h, n); res = []; sel = []
    for tr, te in ((A, B), (B, A)):
        sx = max(inst[x], key=lambda k: inst[x][k][tr].mean()); sz = max(inst[z], key=lambda k: inst[z][k][tr].mean())
        res.append(tp(inst[x][sx][te] - inst[z][sz][te])); sel.append(str(sx))
    return dict(mean=float(np.mean([r["mean"] for r in res])), verdict=verdict(*res), selected=sel)
RES = {}; REF_SRC = {}
for nm in SETS:
    Gr, _ = ckpt(f"gt2_23_{nm}.pkl"); Rr, REF_SRC[nm] = ref_runs(nm); ids = sorted(Gr)
    assert ids == sorted(Rr)[:len(ids)] and len(ids) == n_inst(nm), f"{nm}: instances do not match"
    keys = list(Gr[ids[0]]["rec"])
    inst = {f: inst_means(lambda s, k, f=f: Rr[s]["rec"][f][k]["ari"], ids, list(Rr[ids[0]]["rec"][f])) for f in FAMS}
    inst["GT2"] = inst_means(lambda s, k: Gr[s]["rec"][k]["ari"], ids, keys)
    r = dict(C=Gr[ids[0]]["C"], n=len(ids))
    for x, z in (("GT2", "FCM"), ("GT2", "T1dec"), ("GT2", MM), (MM, "FCM"), (MM, "T1dec")):
        r[f"{x} vs {z}"] = lt_cmp(inst, x, z)
    g = {k: v.mean() for k, v in inst["GT2"].items()}; i2 = {k: v.mean() for k, v in inst[MM].items()}
    r["GT2 best"] = float(max(g.values())); r["GT2 median"] = float(np.median(list(g.values())))
    r["IT2 best"] = float(max(i2.values())); r["IT2 median"] = float(np.median(list(i2.values())))
    r["share GT2 > IT2 at the same setting"] = float(np.mean([g[k] > i2[k] for k in keys]))
    r["converged"] = float(np.mean([c_ for s in ids for k in keys for c_ in Gr[s]["rec"][k]["conv"]]))
    r["runs"] = int(sum(len(Gr[s]["rec"][k]["ari"]) for s in ids for k in keys))
    r["points differing from nearest prototype"] = int(sum(sum(Gr[s]["rec"][k]["mism"]) for s in ids for k in keys))
    sel = r["GT2 vs FCM"]["selected"]
    r["largest class kept, selected settings"] = float(np.mean([np.mean(Gr[s]["rec"][eval(sel[0] if s >= len(ids) // 2 else sel[1])]["kept"]) for s in ids]))
    RES[nm] = r

In [ ]:
LINES = []
def say(s=""): print(s); LINES.append(s)
mk = lambda v: "*" if v == "BETTER" else ("-" if v == "WORSE" else " ")
say(f"23 -- general type-2 variant (alpha-planes 1/3, 2/3, 1) at the settings of IT2   (run {RUN_STAMP})")
say("label-tuned ARI differences x100 (* better, - worse by the claim rule); best/median over the grid; decision check")
for nm in SETS:
    r = RES[nm]
    say(f"  {nm:12s} C={r['C']}  GT2-FCM {100*r['GT2 vs FCM']['mean']:+6.1f}{mk(r['GT2 vs FCM']['verdict'])}  GT2-T1d {100*r['GT2 vs T1dec']['mean']:+6.1f}{mk(r['GT2 vs T1dec']['verdict'])}"
        f"  GT2-IT2 {100*r['GT2 vs ' + MM]['mean']:+6.1f}{mk(r['GT2 vs ' + MM]['verdict'])}  (IT2-FCM {100*r[MM + ' vs FCM']['mean']:+6.1f}{mk(r[MM + ' vs FCM']['verdict'])})"
        f"  GT2 {100*r['GT2 best']:.1f}/{100*r['GT2 median']:.1f}  IT2 {100*r['IT2 best']:.1f}/{100*r['IT2 median']:.1f}  GT2>IT2 at {100*r['share GT2 > IT2 at the same setting']:.0f}% of settings"
        f"  conv {100*r['converged']:.0f}%  points off nearest {r['points differing from nearest prototype']} in {r['runs']} runs  kept {100*r['largest class kept, selected settings']:.0f}%")
two = [nm for nm in SETS if RES[nm]["C"] == 2]
say(f"two-class sets: {sum(RES[nm]['points differing from nearest prototype'] for nm in two)} points off the nearest-prototype partition in "
    f"{sum(RES[nm]['runs'] for nm in two)} runs (Theorem 1(c) requires 0)")
mains = [nm for nm in MAIN if nm in RES]; ACROSS = {}
if len(mains) >= 6:
    for x, z in (("GT2", "FCM"), ("GT2", "T1dec"), ("GT2", MM)):
        d = np.array([RES[nm][f"{x} vs {z}"]["mean"] for nm in mains])
        ACROSS[f"{x} vs {z}"] = dict(n=len(d), median=float(np.median(d)), positive=int((d > 0).sum()), wilcoxon_p=float(stats.wilcoxon(d).pvalue),
                                     better=sum(RES[nm][f"{x} vs {z}"]["verdict"] == "BETTER" for nm in mains), worse=sum(RES[nm][f"{x} vs {z}"]["verdict"] == "WORSE" for nm in mains))
        v = ACROSS[f"{x} vs {z}"]
        say(f"  across {len(mains)} main sets: {x} - {z} median {100*v['median']:+.2f}, positive on {v['positive']}, Wilcoxon p = {v['wilcoxon_p']:.3g}, better on {v['better']}, worse on {v['worse']}")
payload = dict(notebook="23", run_stamp=RUN_STAMP, alphas=list(ALPHAS), reference_runs=REF_SRC, results=RES, across_main=ACROSS,
               two_class_check=dict(sets=two, runs=sum(RES[nm]["runs"] for nm in two), points_off=sum(RES[nm]["points differing from nearest prototype"] for nm in two)))
for p in (os.path.join(OUTDIR, "results_23.json"), os.path.join(OUTDIR, "results_23_report.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n")); fh.flush(); os.fsync(fh.fileno())

In [ ]:
# Check of the iteration limit: GT2 at its label-tuned settings with 1,000 iterations.
CONV_SETS = [nm for nm in ("imb_C5_r20", "imb_C8_r20", "ecoli") if nm in RES]
CONV = {}
for nm in CONV_SETS:
    sel = [eval(k) for k in RES[nm]["GT2 vs FCM"]["selected"]]; n = n_inst(nm); h = n // 2
    ari = {300: [], 1000: []}; conv = {300: 0, 1000: 0}; runs = 0
    for s in range(n):
        a, b, mode = sel[0] if s >= h else sel[1]; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
        X, y, C = inst_any(nm, s); order = np.argsort(X, axis=0, kind="stable")
        for q in range(N_INIT):
            V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s)); runs += 1
            for mx in (300, 1000):
                V, lab, near, it, cv = run_gt2(X, V0, a, b, order, mc, maxit=mx)
                ari[mx].append(scores(y, lab, C)["ari"]); conv[mx] += cv
    CONV[nm] = dict(selected=[str(k) for k in sel], runs=runs, ari_300=float(np.mean(ari[300])), ari_1000=float(np.mean(ari[1000])),
                    converged_300=conv[300] / runs, converged_1000=conv[1000] / runs)
    say(f"  {nm:12s} mean ARI x100 at 300 iterations {100*CONV[nm]['ari_300']:.2f}, at 1,000 {100*CONV[nm]['ari_1000']:.2f} "
        f"(change {100*(CONV[nm]['ari_1000']-CONV[nm]['ari_300']):+.2f}); converged {100*CONV[nm]['converged_300']:.0f}% and {100*CONV[nm]['converged_1000']:.0f}%")
payload["iteration_limit_check"] = CONV
for p in (os.path.join(OUTDIR, "results_23.json"), os.path.join(OUTDIR, "results_23_report.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n")); fh.flush(); os.fsync(fh.fileno())